In [1]:
# ==============================
# Mount Google Drive
# ==============================
from google.colab import drive
drive.mount('/content/drive')

import os

normal_count = len(os.listdir("/content/drive/MyDrive/poaching_project/dataset/frames/normal"))
threat_count = len(os.listdir("/content/drive/MyDrive/poaching_project/dataset/frames/threat"))

print("Normal frames:", normal_count)
print("Threat frames:", threat_count)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import cv2
import os

base_path = "/content/drive/MyDrive/poaching_project/dataset"
video_path = os.path.join(base_path, "videos")
frame_path = os.path.join(base_path, "frames")

TARGET_FPS = 2

def extract_frames_from_videos(label):
    video_folder = os.path.join(video_path, label)
    save_folder = os.path.join(frame_path, label)

    os.makedirs(save_folder, exist_ok=True)

    for video_file in os.listdir(video_folder):
        video_full_path = os.path.join(video_folder, video_file)

        if not video_file.lower().endswith((".mp4", ".avi", ".mov", ".mkv")):
            continue

        print(f"Processing {video_file}")

        cap = cv2.VideoCapture(video_full_path)

        if not cap.isOpened():
            print(f"Skipping {video_file} (cannot open)")
            continue

        original_fps = cap.get(cv2.CAP_PROP_FPS)
        frame_interval = int(original_fps / TARGET_FPS)

        frame_count = 0
        saved_count = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_count % frame_interval == 0:
                frame_name = f"{os.path.splitext(video_file)[0]}_frame_{saved_count:05d}.jpg"
                save_path = os.path.join(save_folder, frame_name)
                cv2.imwrite(save_path, frame)
                saved_count += 1

            frame_count += 1

        cap.release()
        print(f"Saved {saved_count} frames from {video_file}")

    print(f"Done extracting {label} videos.\n")

extract_frames_from_videos("normal")
extract_frames_from_videos("threat")

Processing N2_C1_P1_V1_HB_1.mp4
Saved 11 frames from N2_C1_P1_V1_HB_1.mp4
Processing N2_C1_P2_V1_HB_1.mp4
Saved 21 frames from N2_C1_P2_V1_HB_1.mp4
Processing N1_C1_P2_V1_HB_1.mp4
Saved 17 frames from N1_C1_P2_V1_HB_1.mp4
Processing N1_C1_P4_V1_HB_1.mp4


In [ ]:
!pip install ultralytics --quiet
!pip install tqdm --quiet

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from tqdm import tqdm
import torch

print("CUDA available:", torch.cuda.is_available())

det_model = YOLO("/content/drive/MyDrive/poaching_project/models/yolov8n.pt")
pose_model = YOLO("/content/drive/MyDrive/poaching_project/models/yolov8n-pose.pt")

base_path = "/content/dataset/frames"

JOINT_MAP = {
    "RS": 6, "RE": 8, "RW": 10,
    "LS": 5, "LE": 7, "LW": 9,
    "RH": 12, "RK": 14, "RF": 16,
    "LH": 11, "LK": 13, "LF": 15,
}

def normalize_skeleton(joints):
    hip_center = (joints[6] + joints[9]) / 2
    joints = joints - hip_center

    shoulder_width = np.linalg.norm(joints[0] - joints[3])
    if shoulder_width < 1e-6:
        return None

    return joints.flatten()

In [ ]:
data = []

for label_name in ["normal", "threat"]:
    folder = os.path.join(base_path, label_name)
    label = 0 if label_name == "normal" else 1

    files = os.listdir(folder)

    for file in tqdm(files, desc=f"Processing {label_name}"):

        img_path = os.path.join(folder, file)
        img = cv2.imread(img_path)

        if img is None:
            continue

        det_results = det_model(img, device=0, verbose=False)
        boxes = det_results[0].boxes

        if boxes is None or len(boxes) == 0:
            continue

        person_indices = [i for i, cls in enumerate(boxes.cls) if int(cls) == 0]
        if len(person_indices) == 0:
            continue

        pose_results = pose_model(img, device=0, verbose=False)
        keypoints = pose_results[0].keypoints

        if keypoints is None or keypoints.xy.shape[0] == 0:
            continue

        kp = keypoints.xy[0].cpu().numpy()

        selected = np.array([
            kp[6], kp[8], kp[10],
            kp[5], kp[7], kp[9],
            kp[12], kp[14], kp[16],
            kp[11], kp[13], kp[15],
        ])

        features = normalize_skeleton(selected)
        if features is None:
            continue

        data.append(list(features) + [label])

columns = [f"f{i}" for i in range(24)] + ["label"]
df = pd.DataFrame(data, columns=columns)

save_path = "/content/drive/MyDrive/poaching_project/pose_dataset.csv"
df.to_csv(save_path, index=False)

print("Saved dataset to:", save_path)
print("Total usable samples:", len(df))
print(df["label"].value_counts())

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/poaching_project/pose_dataset.csv")

print("Total samples:", len(df))
print(df["label"].value_counts())
print(df.isnull().sum().sum(), "total missing values")

X = df.drop("label", axis=1).values

print("Min value:", np.min(X))
print("Max value:", np.max(X))
print("Zeros:", np.sum(X == 0))

print("Rows all zero:", np.sum(np.all(X == 0, axis=1)))

mask = np.max(np.abs(X), axis=1) < 50

print("Before:", len(df))
print("After:", np.sum(mask))

df = df[mask].reset_index(drop=True)

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop("label", axis=1).values
y = df["label"].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

In [ ]:
!pip install xgboost --quiet

from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix

model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=1.2,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

In [ ]:
import joblib
joblib.dump(model, "/content/drive/MyDrive/poaching_project/models/xgb_threat_model.pkl")

In [ ]:
model = joblib.load("/content/drive/MyDrive/poaching_project/models/xgb_threat_model.pkl")

sample = np.random.rand(1, 26)
pred = model.predict(sample)

print("Prediction works:", pred)

In [ ]:
# ==============================
# Inference on Single Image
# ==============================

import cv2
import numpy as np
import joblib
from ultralytics import YOLO

# -------------------------
# Load Models
# -------------------------
det_model = YOLO("/content/drive/MyDrive/poaching_project/models/yolov8n.pt")
pose_model = YOLO("/content/drive/MyDrive/poaching_project/models/yolov8n-pose.pt")
clf = joblib.load("/content/drive/MyDrive/poaching_project/models/xgb_threat_model.pkl")

# -------------------------
# Joint Mapping
# -------------------------
JOINT_MAP = {
    "RS": 6, "RE": 8, "RW": 10,
    "LS": 5, "LE": 7, "LW": 9,
    "RH": 12, "RK": 14, "RF": 16,
    "LH": 11, "LK": 13, "LF": 15,
}

# -------------------------
# Helper Functions
# -------------------------
def normalize_skeleton(joints):
    hip_center = (joints[6] + joints[9]) / 2
    joints = joints - hip_center

    shoulder_width = np.linalg.norm(joints[0] - joints[3])
    if shoulder_width < 1e-6:
        return None

    return joints / shoulder_width


def compute_angle(a, b, c):
    ba = a - b
    bc = c - b
    denom = np.linalg.norm(ba) * np.linalg.norm(bc)

    if denom < 1e-6:
        return 0.0

    cosine = np.dot(ba, bc) / denom
    cosine = np.clip(cosine, -1.0, 1.0)
    return np.arccos(cosine)


# -------------------------
# Load Image
# -------------------------
image_path = "/content/drive/MyDrive/poaching_project/gettyimages-2192499195-612x612.jpg"
frame = cv2.imread(image_path)

if frame is None:
    print("❌ Image not found.")

else:
    # -------------------------
    # Step 1: Person Detection
    # -------------------------
    det_results = det_model(frame, device=0, verbose=False)
    boxes = det_results[0].boxes

    if boxes is None or len(boxes) == 0 or 0 not in boxes.cls.cpu().numpy():
        print("❌ No person detected.")

    else:
        # -------------------------
        # Step 2: Pose Estimation
        # -------------------------
        pose_results = pose_model(frame, device=0, verbose=False)
        keypoints = pose_results[0].keypoints

        if keypoints is None or keypoints.xy.shape[0] == 0:
            print("❌ No skeleton detected.")

        else:
            kp = keypoints.xy[0].cpu().numpy()

            # -------------------------
            # Step 3: Select Key Joints
            # -------------------------
            selected = np.array([
                kp[JOINT_MAP["RS"]],
                kp[JOINT_MAP["RE"]],
                kp[JOINT_MAP["RW"]],
                kp[JOINT_MAP["LS"]],
                kp[JOINT_MAP["LE"]],
                kp[JOINT_MAP["LW"]],
                kp[JOINT_MAP["RH"]],
                kp[JOINT_MAP["RK"]],
                kp[JOINT_MAP["RF"]],
                kp[JOINT_MAP["LH"]],
                kp[JOINT_MAP["LK"]],
                kp[JOINT_MAP["LF"]],
            ])

            # -------------------------
            # Step 4: Normalize
            # -------------------------
            normalized = normalize_skeleton(selected)

            if normalized is None:
                print("❌ Bad pose normalization.")

            else:
                flat = normalized.flatten()

                # -------------------------
                # Step 5: Angle Features
                # -------------------------
                RS, RE, RW = normalized[0], normalized[1], normalized[2]
                LS, LE, LW = normalized[3], normalized[4], normalized[5]

                right_angle = compute_angle(RS, RE, RW)
                left_angle = compute_angle(LS, LE, LW)

                features = np.concatenate([flat, [right_angle, left_angle]])

                # -------------------------
                # Step 6: Prediction
                # -------------------------
                prob = clf.predict_proba(features.reshape(1, -1))[0][1]
                pred = 1 if prob > 0.45 else 0

                print(f"Threat probability: {prob:.3f}")
                print("⚠️ THREAT DETECTED" if pred == 1 else "✅ SAFE")

In [ ]:
# ==============================
# Model Evaluation Visualization
# ==============================

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# -------------------------
# Predict on Test Data
# -------------------------
y_pred = model.predict(X_test)

# -------------------------
# Classification Report
# -------------------------
print("=== Classification Report ===\n")
print(classification_report(y_test, y_pred, digits=4))

# -------------------------
# Confusion Matrix
# -------------------------
cm = confusion_matrix(y_test, y_pred)

print("=== Confusion Matrix ===")
print(cm)

# -------------------------
# Visualization
# -------------------------
plt.figure(figsize=(5, 4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Predicted Normal", "Predicted Threat"],
    yticklabels=["Actual Normal", "Actual Threat"]
)

plt.xlabel("Prediction")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Threat Classifier")

plt.show()